# Advance Algorithm in Machine Learning

## Q1: SVM on Diabetes Dataset

Author: Souvik Karmakar & Arif Bin Azhar
Roll No: BIM-2024-25 & BIM-2024-06

### Loading Libraries

In [3]:
import pandas as pd
import numpy as np

from sklearn.datasets import load_diabetes
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

### Loading Datasets

In [4]:
data = load_diabetes()

X = data.data
y = data.target

# Convert to binary classification
median_target = np.median(y)
y = (y > median_target).astype(int)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC())
])


### Parameter Grid

In [5]:
param_grid = [
    {
        'svm__kernel': ['linear'],
        'svm__C': [0.1, 1, 10]
    },
    {
        'svm__kernel': ['rbf'],
        'svm__C': [0.1, 1, 10],
        'svm__gamma': ['scale', 0.01, 0.1]
    },
    {
        'svm__kernel': ['poly'],
        'svm__C': [0.1, 1, 10],
        'svm__gamma': ['scale', 0.01],
        'svm__degree': [2, 3, 4]
    }
]


### 5 fold CV

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

grid.fit(X, y)

print("Best Parameters:", grid.best_params_)
print("Best Accuracy:", grid.best_score_)


Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best Parameters: {'svm__C': 1, 'svm__gamma': 0.01, 'svm__kernel': 'rbf'}
Best Accuracy: 0.7600612870275791


In [7]:

results = pd.DataFrame(grid.cv_results_)

# Select useful columns
cols = [
    'param_svm__kernel',
    'param_svm__C',
    'param_svm__gamma',
    'param_svm__degree',
    'mean_test_score'
]

results = results[cols]
results.rename(columns={
    'param_svm__kernel': 'Kernel',
    'param_svm__C': 'C',
    'param_svm__gamma': 'Gamma',
    'param_svm__degree': 'Degree',
    'mean_test_score': 'Accuracy'
}, inplace=True)

results.to_csv("svm_diabetes_results.csv", index=False)

print("Results saved as svm_diabetes_results.csv")




Results saved as svm_diabetes_results.csv


### Detailed Metrics on Best Model

In [8]:


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

best_model = grid.best_estimator_
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

print("\nFinal Test Metrics:")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))


Final Test Metrics:
Accuracy : 0.7303370786516854
Precision: 0.717391304347826
Recall   : 0.75
F1-score : 0.7333333333333333
